# Level 3 · Harnesses, modes & the full reference

Levels 1 and 2 each had one job. This notebook has none — it is the reference you come back to,
organised so you can jump to the part you need:

| | |
|---|---|
| **§1 Standard evaluation** | use this like any ordinary agentic benchmark, with none of the evolving machinery |
| **§2 The provided harnesses** | `react` and `codex`, every option, and how they fail |
| **§3 The three modes** | what each one holds fixed, and which resource pool it implies |
| **§4 The evolving axis** | inspecting tools / skills / agents at any stage |
| **§5 ALE in depth** | inputs, artifacts, the run tree, Codex-in-the-sandbox |
| **§6 Persistent memory** | `memory_key` and consolidation |
| **§7 Long runs** | timeouts, backgrounding, what survives a dropped connection |
| **§8 Observability** | usage, the dashboard, client logging |
| **§9 Raw HTTP** | every call without the SDK |
| **§10 API reference** | the complete dictionary |

### The three tutorials

| | | |
|---|---|---|
| **1 · Leaderboard your agent** | your method → one comparable row | [Colab](https://colab.research.google.com/drive/1xJEpRf_s0zG-M9QynS3MBk7Nkr-xB11r) |
| **2 · Continual learning** | the matrix, retention, BWT and FWT | [Colab](https://colab.research.google.com/drive/1vrcGelN9GmwiCK25c6qZNG3Z0sHWxE5o) |
| **3 · Harnesses & modes** | everything, as a reference | [Colab](https://colab.research.google.com/drive/1mYQEDCVFStXMRWYI2hpEGBXwyRx1NSFj) ← **you are here** |

Setup and the health check are identical to Level 1. Run them, then jump anywhere.

In [ ]:
# Setup, in one cell. The client SDK is served BY the service (GET /sdk), so there is no
# PyPI account, no repo checkout, and no service URL anywhere in your code afterwards.
import getpass, importlib, json, os, re, subprocess, sys, tempfile, urllib.error, urllib.request
from functools import partial

SERVICE_URL = os.environ.get("EVAL_SERVICE_URL",
                             "https://educator-marrow-cultural.ngrok-free.dev")

if not os.environ.get("EVAL_SERVICE_API_KEY"):
    os.environ["EVAL_SERVICE_API_KEY"] = getpass.getpass("Eval service key (MyAuthtoken): ")
SDK_HEADERS = {"Authorization": f"Bearer {os.environ['EVAL_SERVICE_API_KEY']}",
               "ngrok-skip-browser-warning": "true"}
def sdk_read(request):
    try:
        with urllib.request.urlopen(request) as response:
            return response.read()
    except urllib.error.HTTPError as exc:
        try: detail = json.loads(exc.read()).get("detail", str(exc))
        except Exception: detail = str(exc)
        raise SystemExit(f"SDK download denied: {detail}") from None


# Always load the exact wheel advertised by this deployment into an isolated directory.
# Putting it first on sys.path prevents an older editable/repo copy from shadowing it.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "openai", "httpx"], check=True)
req = urllib.request.Request(f"{SERVICE_URL}/sdk", headers=SDK_HEADERS)
sdk_manifest = json.loads(sdk_read(req))
sdk_root = tempfile.mkdtemp(prefix="simple_agentic_evals_sdk_")
sdk_target = os.path.join(sdk_root, "site")
sdk_wheel = os.path.join(sdk_root, sdk_manifest["filename"])
wheel_req = urllib.request.Request(f"{SERVICE_URL}{sdk_manifest['path']}", headers=SDK_HEADERS)
with open(sdk_wheel, "wb") as output:
    output.write(sdk_read(wheel_req))
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--force-reinstall", "--no-cache-dir", "--no-deps",
                "--target", sdk_target, sdk_wheel],
               check=True)
sys.path.insert(0, sdk_target)
for k in [k for k in list(sys.modules) if k.startswith("simple_agentic_evals")]:
    del sys.modules[k]
importlib.invalidate_caches()
import simple_agentic_evals as _eval_sdk
served_sdk = re.search(r"-(\d+\.\d+\.\d+)-", sdk_manifest["filename"]).group(1)
probe = _eval_sdk.Task(None, {}, "_sdk_probe")
if (_eval_sdk.__version__ != served_sdk or not hasattr(probe, "required_steps")
        or not hasattr(probe, "evaluation")):
    raise RuntimeError("served SDK failed its Task metadata check")
print("sdk ->", _eval_sdk.__version__, _eval_sdk.__file__)

if not os.environ.get("OPENAI_API_KEY"):        # the key YOUR agent thinks with
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from openai import OpenAI
from simple_agentic_evals import (EvalClient, ServiceError, run_benchmark,
                                  react_agent, acp_codex_agent, to_openai_tools)

client    = EvalClient()     # the endpoint is baked into the wheel -- no URL to paste
oai       = OpenAI()         # only for the bring-your-own-agent examples
LLM_MODEL = os.environ.get("EVAL_LLM_MODEL", "gpt-4o-mini")
# Every /v1/* route needs the key, health included -- a rejected key means you
# cannot use this service, so it should not report itself healthy. So this one
# call both connects and proves the key, and a bad key fails HERE with the
# message that says where to get a new one.
try:
    _h = client.health()
except ServiceError as e:
    raise SystemExit(f"cannot use the eval service: {e.detail}") from None
print("connected ->", _h)


def harness(fn, *args, **kwargs):
    """Call a provided harness, or say why this deployment can't and return None.

    react_agent / acp_codex_agent execute ON THE SERVICE, so they depend on what that host
    has installed. Bringing your own agent needs none of it.
    """
    try:
        return fn(*args, **kwargs)
    except ServiceError as e:
        # The service prefixes a banner and appends the subprocess's stderr tail, so the
        # LAST line is the actual cause -- the first is just "...stderr tail:".
        lines = [ln for ln in (e.detail or "").strip().splitlines() if ln.strip()]
        print(f"  [harness unavailable] {e.status_code}: {(lines[-1] if lines else e)!s:.170}")
        if len(lines) > 1:
            print(f"     ({len(lines)} more lines of server traceback in e.detail)")
        return None

## 0. Health check — run this first

The service is a **deployment**, not a library. The environment, the graders and the two
provided harnesses all execute on *that host*, so what actually works depends on what it has
installed. The failure mode is quiet: an unavailable harness returns **without acting**, and
its run then scores the do-nothing baseline — which reads as a real (bad) result rather than
an error. Verify the deployment before you trust a number from it.

Each row is one capability, and a red row tells you what you lose, not what you did wrong:

| row | what it proves | if it's red |
|---|---|---|
| `reachable`, `sdk` | service is up; your client matches its served wheel | nothing works / re-run Setup |
| `catalog`, `eog tasks` | both benchmarks advertised, the slice is populated *and its count matches the catalog* | that data isn't deployed |
| `ale tasks` | the ALE **denominator**: exactly the tasks a published number counts | you can't compare an ALE score to anything |
| `eog env+grade` | session provisioning, the gym MCP proxy, SQL verifiers | all of EOG (§2) |
| `resources` | the evolving axis — `oracle` ⊆ `accumulative` | the evolving modes (§3, §4) |
| `react`, `codex` | the **server-side** harnesses, and the host packages they need | those harnesses only — your own loop needs neither |
| `ale` | input staging, artifact submit, the task's own `evaluate()` | all of ALE (§5) |

The harness rows are the only way to spot a server missing `langchain` or the Codex CLI. A
harness that *runs* but stops with `error` is counted **red** — that's the silent failure this
section exists to catch. Set `DEEP = False` to skip the three rows that cost LLM calls or boot
a sandbox.

The task-count rows are about a subtler kind of wrong. ALE ships 152 tasks, but 47 of them need
a non-Linux VM and a few more die in their own loader on any given host, so only a subset can
produce a number at all — and a task that provisions and returns `0.0` for an environmental
reason is indistinguishable from an agent that tried and failed. The service therefore serves
only the tasks its exclusion manifest says are measurable, and `ale tasks` re-derives that
count from the catalog and fails if the two disagree. A green row means an ALE score you can
put next to someone else's.

In [ ]:
import urllib.error
DEEP = True    # False -> skip the rows that cost LLM calls or boot an ALE sandbox

# Fixed reference slices, so this stays a self-test of the DEPLOYMENT whatever you set in 1.
EOG_PROBE = ("evovling_tools", "eog", 1, "test", "hr")   # dataset, benchmark, version, split, domain
ALE_PROBE = ("evovling_tools", "ale", 1, "legal/agora_governance_classify_instance_1")
_rows = []


def _check(name, fn, lose="", deep=False):
    if deep and not DEEP:
        _rows.append((name, "SKIP"))
        print(f"  [SKIP] {name:14} DEEP=False")
        return
    try:
        detail, status = fn(), "OK"
    except ServiceError as e:                    # the service said no -- report why
        lines = [ln for ln in (e.detail or "").strip().splitlines() if ln.strip()]
        detail, status = f"{e.status_code}: {(lines[-1] if lines else e)!s:.100}", "FAIL"
    except Exception as e:
        detail, status = f"{type(e).__name__}: {e!s:.100}", "FAIL"
    _rows.append((name, status))
    print(f"  [{status:4}] {name:14} {detail}")
    if status == "FAIL" and lose:
        print(f"         lose: {lose}")


def _one_eog():
    return next(client.tasks(*EOG_PROBE, limit=1))


def _auth():
    """Prove the key works before anything expensive depends on it.

    A rejected key fails every row below this one, all with errors that look like
    the deployment is broken rather than like a credential problem -- so name it here.
    """
    req = urllib.request.Request(f"{SERVICE_URL}/v1/health",
                                 headers={"ngrok-skip-browser-warning": "true"})
    try:                                          # no key at all: is the gate even on?
        urllib.request.urlopen(req, timeout=30)
        gated = False
    except urllib.error.HTTPError as e:
        if e.code not in (401, 403, 503):
            raise
        gated = True
    client.benchmarks()                           # now WITH the key -- raises if rejected
    return ("key accepted" if gated else
            "key accepted (deployment is currently open -- no key required)")


def _service():
    h = client.health()
    if not h.get("ok"):
        raise RuntimeError(f"health.ok={h.get('ok')!r}")
    return f"ok, {h.get('active_sessions')} live session(s), ttl {h.get('ttl_sec')}s"


def _sdk():
    import simple_agentic_evals as m
    req = urllib.request.Request(f"{SERVICE_URL}/sdk",
                                 headers={"Authorization": f"Bearer {os.environ['EVAL_SERVICE_API_KEY']}",
                                          "ngrok-skip-browser-warning": "true"})
    served = re.search(r"-(\d+\.\d+\.\d+)-",
                       json.loads(sdk_read(req))["filename"]).group(1)
    mine = str(getattr(m, "__version__", "?"))
    if mine != served:                           # a stale client drifts from the API silently
        raise RuntimeError(f"client {mine} != served {served} -- re-run Setup")
    return f"{mine}, matches the served wheel"


def _catalog():
    kinds = {b["kind"] for ds in client.benchmarks()["datasets"]
             for b in ds["benchmarks"] if b.get("kind") in ("eog", "ale")}
    if {"eog", "ale"} - kinds:
        raise RuntimeError(f"catalog is missing {sorted({'eog', 'ale'} - kinds)}")
    return "eog + ale both advertised"


def _advertised(dataset, benchmark, version, split, domain):
    """What the catalog claims a stage holds, for cross-checking against task_ids."""
    for ds in client.benchmarks()["datasets"]:
        if ds["dataset"] != dataset:
            continue
        for b in ds["benchmarks"]:
            if b["benchmark"] != benchmark:
                continue
            for d in b["domains"]:
                if d["domain"] != domain:
                    continue
                for v in d["versions"]:
                    if v["version"] == version:
                        return v[f"n_{split}"]
    return None


def _eog_tasks():
    ids = client.task_ids(*EOG_PROBE)
    if not ids:
        raise RuntimeError("reference slice is empty")
    n = _advertised(*EOG_PROBE)                  # catalog and listing must agree
    if n != len(ids):
        raise RuntimeError(f"catalog advertises {n} tasks, task_ids returns {len(ids)}")
    return f"{len(ids)} tasks in {'/'.join(map(str, EOG_PROBE[1:]))}, matches the catalog"


def _ale_tasks():
    """ALE serves only what a published number can count -- verify the arithmetic."""
    m = client.health().get("ale_tasks") or {}
    if not m.get("ok"):
        raise RuntimeError(f"ALE task set unavailable: {m.get('reason')}")
    if not m.get("manifest_agrees"):
        raise RuntimeError(f"manifest drifted from ALE's task lists: {m.get('disagreements')}")
    if not m.get("filtered"):
        raise RuntimeError("this deployment sets EVAL_SERVICE_ALE_SERVE_ALL, so the catalog "
                           "includes tasks no published ALE number counts")
    served = set()
    for ds in client.benchmarks()["datasets"]:
        for b in ds["benchmarks"]:
            if b.get("kind") != "ale":
                continue
            for d in b["domains"]:
                for v in [x["version"] for x in d["versions"]]:
                    for sp in ("train", "test"):
                        served |= set(client.task_ids(ds["dataset"], "ale", v, sp,
                                                      d["domain"] or None))
    if len(served) != m["n_runnable"]:
        raise RuntimeError(f"catalog lists {len(served)} ale tasks, "
                           f"the manifest says {m['n_runnable']} are runnable")
    if m.get("example_filtered") in served:      # prove the filter is actually live
        raise RuntimeError(f"withheld task {m['example_filtered']} is still listed")
    return (f"{len(served)}/{m['n_suite']} runnable = {m['n_docker_support']} docker + "
            f"{m['n_privileged']} privileged; {m['n_excluded']} excluded + "
            f"{m['n_not_linux']} non-Linux withheld")


def _eog_env():
    task = _one_eog()
    with task:                                   # fresh DB, freed on exit
        mcp = task.mcp_session(task.mcp_servers[0])
        try:
            tools = mcp.list_tools()
        finally:
            mcp.close()
        if not tools:
            raise RuntimeError("gym returned 0 MCP tools")
        g = task.grade(keep_alive=True)          # no agent acted -> this is the baseline
    if not g.n_total:
        raise RuntimeError("grader ran 0 verifiers")
    return f"{len(tools)} tools, {g.n_total} verifiers, 1-task baseline {g.pass_rate:.2f}"


def _resources():
    tid = client.task_ids(*EOG_PROBE)[0]
    c = {m: client.resources(*EOG_PROBE[:3], task_id=tid, split=EOG_PROBE[3],
                             domain=EOG_PROBE[4], mode=m)["count"]
         for m in ("none", "oracle", "accumulative")}
    if c["accumulative"] < c["oracle"]:
        raise RuntimeError(f"accumulative {c['accumulative']} < oracle {c['oracle']}")
    return "  ".join(f"{k}={v}" for k, v in c.items())


def _harness_row(fn, **kw):
    task = _one_eog()
    with task:
        run = fn(task, api_key=os.environ["OPENAI_API_KEY"], **kw)
    stopped = getattr(run, "stopped", "?")
    if stopped == "error":                       # ran, but the turn failed on the server
        raise RuntimeError("harness returned stopped='error' -- see the service log")
    return f"ran on the service (stopped={stopped})"


def _ale():
    ad = client.health().get("ale_docker") or {}
    task = client.task(*ALE_PROBE, domain=None)  # ALE is flat -> domain=None
    with task:
        files = task.inputs()
        if not files:
            raise RuntimeError("no input files staged")
        task.fetch_input(files[0]["path"])
        # A stub artifact is enough: we're proving evaluate() executes, not scoring well.
        task.submit_text(task.output_path or "output/agent_output.json", "{}")
        g = task.grade(keep_alive=True)
    return (f"{len(files)} inputs, evaluate() ran (stub -> {g.pass_rate}), "
            f"sandbox={ad.get('enabled')} dind={ad.get('dind_available')}")


print("deployment:", SERVICE_URL)
_check("auth",          _auth,      "everything -- get a key from MyAuthtoken")
_check("reachable",     _service,   "everything")
_check("sdk",           _sdk,       "silent API drift -- re-run Setup")
_check("catalog",       _catalog,   "a benchmark isn't deployed here")
_check("eog tasks",     _eog_tasks, "this EOG slice")
_check("ale tasks",     _ale_tasks, "a trustworthy ALE denominator (5)")
_check("eog env+grade", _eog_env,   "all of EOG (2)")
_check("resources",     _resources, "the evolving modes (6)")
_check("react",         lambda: _harness_row(react_agent, model=LLM_MODEL, max_steps=1),
       "the ReAct harness -- your own loop (2) still works", deep=True)
_check("codex",         lambda: _harness_row(acp_codex_agent, max_episodes=1),
       "the Codex harness -- your own loop (2) still works", deep=True)
_check("ale",           _ale,       "all of ALE (5)", deep=True)

skip = [n for n, s in _rows if s == "SKIP"]
red = [n for n, s in _rows if s == "FAIL"]
ok = sum(1 for _, s in _rows if s == "OK")
line = f"\n{ok}/{len(_rows) - len(skip)} green"
line += f"   RED: {', '.join(red)}" if red else "   every feature on this deployment works"
print(line + (f"   (skipped: {', '.join(skip)})" if skip else ""))

## 1. Standard evaluation — none of the evolving machinery

The benchmark is *about* harness growth, but nothing forces you to engage with that. Underneath
is an ordinary agentic benchmark — tasks, an environment, hidden verifiers — and you can use it
exactly that way.

Three levels of "standard", from most to least ordinary:

| what you want | how |
|---|---|
| one number over every task, final harness | `run_benchmark(agent, "deployment_eval")` — Level 1 |
| a fixed slice you control | `client.tasks(dataset, benchmark, version, split, domain)` and loop |
| no evolving resources at all | add `resource_mode="none"` |

The third is the one people mean by "just evaluate my agent." `version=` pins a single stage
instead of sweeping, and `resource_mode="none"` detaches the evolving pool so the agent sees the
gym's base surface only. Nothing accumulates, nothing grows, and no stage ever changes.

`version="full"` is the opposite end: every stage pooled into one flat task set under the final
harness. That is what `run_benchmark` uses for its last row.

### The loop underneath everything

Every path in every notebook bottoms out here:

```python
task = client.task(dataset, benchmark, version, task_id, split=..., domain=...)
with task:              # provision: fresh gym DB (EOG) or staged sandbox (ALE)
    ...                 # act
    grade = task.grade()   # verifiers read the environment; session torn down
```

`with task:` is the whole lifecycle. `grade(keep_alive=True)` keeps the session up afterwards so
you can inspect what your agent did — which is how the debugging in §5 works.

In [ ]:
# The plainest possible evaluation: one fixed stage, no evolving resources.
tid = client.task_ids("evovling_tools", "eog", 1, "test", "hr")[0]
task = client.task("evovling_tools", "eog", 1, tid, split="test", domain="hr",
                   resource_mode="none")

with task:
    print("task_id       ", task.task_id)
    print("benchmark     ", task.benchmark, "| action surface:", task.action_type)
    print("mcp servers   ", [s.name for s in task.mcp_servers])
    print("resources     ", task.resources or "(none attached)")
    print("prompt        ", (task.user_prompt or "")[:110], "...")

    grade = task.grade(keep_alive=True)          # nobody acted -> the do-nothing floor
    print(f"\nfloor         {grade.n_passed}/{grade.n_total} verifiers, "
          f"pass_rate {grade.pass_rate:.3f}")
    for v in grade.per_verifier[:3]:
        print(f"   {'PASS' if v.get('passed') else 'FAIL'}  {str(v.get('name'))[:64]}")

# Compare the pools: this is the only thing "evolving" changes about a task.
counts = {m: client.resources("evovling_tools", "eog", 1, task_id=tid, split="test",
                              domain="hr", mode=m)["count"]
          for m in ("none", "oracle", "accumulative")}
print("\nresource pool by mode:", counts)

## 2. The provided harnesses

Two reference agents run **on the service**, so you need no local model runner, gym, sandbox or
`codex` binary. They are what the paper's baseline rows use.

| | `"react"` | `"codex"` |
|---|---|---|
| what it is | EnterpriseOps-Gym's reference ReAct loop | Codex driven over ACP |
| benchmarks | **EOG only** — an ALE task raises | EOG **and** ALE |
| the loop | reason → call gym tools → repeat to `max_steps` | episodes until a `TASK_COMPLETE` sentinel, up to `max_episodes` |
| on ALE | n/a | drives the Codex CLI inside the sandbox, which solves *and* grades |

Both **require your OpenAI key**: the service hosts the environment, the harness and the grader,
but your inference is yours to pay for, and it will not lend its own credentials. Omit the key and
the SDK raises `MissingAPIKey` *before* it sends anything, so the message lands next to the call
that is missing it rather than as a `400` from somewhere else.

### Three ways to invoke one

```python
react_agent(task, api_key=..., model=..., max_steps=8)   # act only; you grade
task.evaluate(agent="react", api_key=...)                # act + grade, all metrics
run_benchmark("react", agent_kwargs={"api_key": ...})    # a whole sweep
```

`task.evaluate(...)` is usually what you want: one call returns accuracy *and* `latency_s` and
`total_tokens`, with the raw `run` and `grade` attached for anything else.

### Options

| `react_agent` | |
|---|---|
| `model`, `api_key` | which model, whose key |
| `max_steps` | reason/act iterations before it stops |
| `restrict_to_selected_tools` | limit to the task's gold tool set (oracle behaviour) |
| `timeout_s` | wall clock for the whole run (default 1800) |
| `verbose` | `False` \| `"summary"` \| `"steps"` \| `"full"` |
| `include_trace` | force the structured per-step trace on/off |

| `acp_codex_agent` | |
|---|---|
| `model`, `api_key`, `timeout_s` | as above |
| `max_episodes` | re-drives with a nudge until the sentinel (default 4) |
| `require_completion`, `completion_sentinel` | what counts as finished |
| `allowed_tools`, `mcp_only` | restrict the tool surface |
| `prompt_suffix`, `sandbox_env` | extra instructions; env vars inside the sandbox |
| `memory_key`, `memory_main_only` | persistent `CODEX_HOME` across runs — §6 |

### What they cost, and how a run ends

The two are not interchangeable in price. On the same HR task with `gpt-4o-mini`, measured:

| | steps / episodes | wall clock | tokens |
|---|---|---|---|
| `react`, `max_steps=3` | 3 steps, 5 tool calls | ~7 s | ~63 k |
| `codex`, `max_episodes=1` | 1 episode, 249 tool calls | ~410 s | ~8 M |

Codex is two orders of magnitude heavier because it is an agentic CLI that explores; React is a
bounded reason/act loop. Size a sweep accordingly — Codex over a full domain is an overnight job.

`run.stopped` is how you tell a finished run from a truncated one, and it is worth checking
before you believe a score:

| `stopped` | meaning |
|---|---|
| `"done"` | the agent decided it was finished |
| `"max_episodes"` | Codex ran out of episodes without signalling the sentinel |
| `"timeout"` | `timeout_s` expired |
| `"error"` | it failed — treat as no result, not as a zero |

The Codex row above is one of these: it stopped at `max_episodes` with `completed=False`, having
worked for the full budget without ever declaring itself done. That is a real outcome rather than a
bug, but a score attached to it says as much about the budget as about the method.

### When a harness is unavailable

They execute on *that host*, so what works depends on what it has installed. The failure is
quiet in the worst way: an unavailable harness can return **without acting**, and grading then
reports the do-nothing floor — a plausible-looking bad score rather than an error.

Three things catch it. The health check in §0 is the first. The second is that `run_benchmark`
treats `run.stopped == "error"` as a failed task rather than a real zero, and counts it in
`report.n_errors` — so a sweep whose `n_errors` is not zero has not measured what you think, and
`on_error="raise"` will stop on the first one instead. The third is `MissingAPIKey`, which is the
one failure that always propagates: a forgotten key would otherwise score a clean-looking 0.0
across every task in the sweep.

In [ ]:
OPENAI_KEY = os.environ["OPENAI_API_KEY"]

# (a) react_agent -- act only, then grade yourself. Note the trace.
t = next(client.tasks("evovling_tools", "eog", 1, "test", "hr", limit=1))
with t:
    run = harness(react_agent, t, api_key=OPENAI_KEY, model=LLM_MODEL,
                  max_steps=3, include_trace=True)
    if run is not None:
        print(f"react: stopped={run.stopped} steps={run.steps} calls={run.n_calls} "
              f"latency={run.latency_s}s tokens={run.total_tokens}")
        print("tools used:", run.tools_used[:6])
        # A trace step is {thought, tool_calls, tool_results, usage} -- the act loop.
        for i, step in enumerate(run.trace or [], 1):
            calls = ", ".join(c["name"] for c in step.get("tool_calls") or [])
            print(f"   step {i}: {calls or '(no tool call)'}"
                  f"{'  // ' + str(step['thought'])[:48] if step.get('thought') else ''}")
        print("grade:", t.grade(keep_alive=True).pass_rate)

# (b) task.evaluate -- act AND grade in one call, every metric together.
t2 = next(client.tasks("evovling_tools", "eog", 1, "test", "hr", limit=1))
with t2:
    try:
        rep = t2.evaluate(agent="react", api_key=OPENAI_KEY, model=LLM_MODEL,
                          max_steps=3, keep_alive=True)
        print(f"\nevaluate: acc={rep.accuracy:.3f} solved={rep.overall_success} "
              f"latency={rep.latency_s}s tokens={rep.total_tokens}")
    except ServiceError as e:
        print("\nevaluate unavailable:", str(e.detail)[:120])

# (c) react on ALE raises a clear, immediate error -- no session or key needed.
ale = client.task("evovling_tools", "ale", 1,
                  "legal/agora_governance_classify_instance_1", domain=None)
try:
    react_agent(ale, api_key="unused")
except ServiceError as e:
    print("\nreact on ALE ->", str(e.detail)[:100])

In [ ]:
# Codex is the heavyweight: minutes and millions of tokens per task (see the table above),
# so it is off by default. Flip the flag when you actually want to pay for a run.
RUN_CODEX = False

if RUN_CODEX:
    t3 = next(client.tasks("evovling_tools", "eog", 1, "test", "hr", limit=1))
    with t3:
        run = harness(acp_codex_agent, t3, api_key=OPENAI_KEY, model=LLM_MODEL,
                      max_episodes=1, timeout_s=420)
        if run is not None:
            print(f"codex: stopped={run.stopped} completed={run.completed} "
                  f"episodes={run.episodes} calls={run.n_calls}")
            print(f"       latency={run.latency_s:.0f}s tokens={run.total_tokens:,}")
            print(f"       {str(run.final_message)[:150]}")
            print("grade:", t3.grade(keep_alive=True).pass_rate)
else:
    print("RUN_CODEX = False -- skipped (a single Codex task runs for minutes)")

# On ALE, Codex is the only provided option, and it both solves AND grades inside the
# sandbox: ale_run scores the task, so evaluate() returns the score without a separate
# grade() round-trip. cost_usd / cache_read_tokens / n_steps are populated on ALE only.
print("\nALE via codex:  task.evaluate(agent='codex', api_key=..., timeout_s=1800)")

## 3. The three modes

A mode is a statement about **what is allowed to change**. Each one also fixes the resource pool,
so you never set `resource_mode` yourself when using `run_benchmark`.

| `mode=` | harness | your method | resource pool | answers |
|---|---|---|---|---|
| `"deployment_eval"` | grows | **fixed** | `accumulative` | does a frozen method survive growth? |
| `"self_evolving_adapt_eval"` | grows | **adapts** per stage | `accumulative` | does learning keep up with growth? |
| `"task_specific"` | n/a | fixed | `oracle` | how well would it do with only the right capabilities? |

`task_specific` is a **reference condition**, not a third continual-learning setting. Each task
is given only its own annotated capabilities, so there are no distractors and no stages to sweep
— one pass, one number. Asking for `matrix=True` with it is an error rather than a slow surprise.

Aliases are accepted for all of them (`"deployment"`, `"deploy"`, `"adapt"`, `"oracle"`, and the
repo's habitual `"evovling"` spelling), so you rarely have to look up the exact string.

Read together, the three bracket a result: `task_specific` is roughly the ceiling that perfect
capability selection would buy, `deployment_eval` is what actually happens under a growing
catalog, and the gap between them is the cost of distraction.

In [ ]:
from simple_agentic_evals.benchmark import _MODES, _RESOURCE_FOR_MODE

print("aliases -> canonical mode:")
for alias, canon in sorted(_MODES.items()):
    print(f"   {alias:28} -> {canon}")
print("\nmode -> resource pool it implies:")
for m, r in _RESOURCE_FOR_MODE.items():
    print(f"   {m:16} -> {r}")


def noop(task):
    """Free: shows what each mode provisions without paying for inference."""
    return {"total_tokens": 0}


for mode in ("deployment_eval", "task_specific"):
    r = run_benchmark(noop, mode, client=client, domain="hr", limit=1, progress=False)
    print(f"\n{mode:24} resource={r.resource_mode:14} ACC={r.accuracy:.3f} "
          f"n={r.n_tasks}")

## 4. The evolving axis — inspecting it directly

`client.resources(...)` reads the pool for any task at any stage **without provisioning a
session**, so it is the cheap way to see what a run will actually offer the agent before paying
for one.

Three datasets evolve three different things, and they do not cover the same ground:

| dataset | what grows | EOG domains | payload |
|---|---|---|---|
| `evovling_tools` | the tool catalog | all 9 | tool names and schemas |
| `evovling_skills` | the skill library | csm, hr, itsm, tri-hybrid | `SKILL.md` bodies |
| `evovling_agents` | the specialist roster | csm, hr, itsm, tri-hybrid | agent `.toml` definitions |

That coverage difference matters when you pick a domain: `calendar` and `email` exist for tools
only, so a skills run there is a `404`, not an empty result.

`version=` takes a stage number or `"full"` (the whole universe). `mode=` takes `oracle` /
`accumulative` / `none`, and `include_content=True` inlines the bodies. The invariant worth
remembering is `oracle ⊆ accumulative` — the difference between them is exactly the distractor
load, and it is what grows from stage to stage.

In [ ]:
tid = client.task_ids("evovling_tools", "eog", 2, "test", "hr")[0]
for mode in ("none", "oracle", "accumulative"):
    res = client.resources("evovling_tools", "eog", 2, task_id=tid, split="test",
                           domain="hr", mode=mode)
    print(f"  stage 2 {mode:14} count={res['count']:<4} e.g. {(res.get('names') or [])[:5]}")

full = client.resources("evovling_tools", "eog", "full", task_id=tid, split="test",
                        domain="hr", mode="accumulative")
print(f"  full universe            count={full['count']}")

# Skills and agents carry their bodies inline, under items[i]["files"][j]["content"] --
# this is literally the text the agent is handed. `note` says what the pool represents.
for ds in ("evovling_skills", "evovling_agents"):
    r = client.resources(ds, "eog", 1, split="test", domain="hr",
                         mode="accumulative", include_content=True)
    print(f"\n  {ds}: kind={r.get('kind')} count={r.get('count')}")
    print(f"     note: {str(r.get('note'))[:96]}")
    for item in (r.get("items") or [])[:1]:
        f = (item.get("files") or [{}])[0]
        print(f"     {item.get('name')}  ({str(f.get('path')).rsplit('/', 1)[-1]}, "
              f"{f.get('size')} bytes)")
        print(f"     {' '.join(str(f.get('content') or '').split())[:150]}...")

## 5. ALE in depth

ALE uses a **file sandbox**: the service stages inputs, your agent writes outputs, and the
task's own `evaluate()` scores them.

| call | what it does |
|---|---|
| `task.inputs()` | list the staged files (`path`, `size`, …) |
| `task.fetch_input(path)` | one file's bytes |
| `task.fetch_inputs_to(dir)` | all of them onto your disk, relative paths kept |
| `task.output_path` | the deliverable path the prompt asks for |
| `task.required_steps` | ordered public `agentMustDo` steps from the ALE task card |
| `task.evaluation` | public natural-language evaluation rubric and scoring method |
| `task.submit_text(path, text)` | one text artifact |
| `task.submit(files)` | many; `content` for text, `content_b64` for binary |
| `task.submit_dir(dir)` | everything under a directory, encoding chosen per file |
| `task.fetch_run_artifacts()` | the raw `ale_run` tree as tar.gz |

`required_steps` and `evaluation` are public task-card metadata; measured results are in
`grade().per_verifier`. Hidden references are never included.

Scores are continuous values in `[0, 1]`. Named components appear in `per_verifier`, with
low-level checks in `details.checks`. Scalar evaluators return `ale_score`; an early artifact
failure returns `evaluation_preconditions`. `overall_success` means `score >= 1.0`.

Fetch run artifacts before `grade()`, or use `evaluate(fetch_artifacts=True)`.

In [ ]:
task = client.task("evovling_tools", "ale", 1,
                   "legal/agora_governance_classify_instance_1", domain=None)
with task:
    print("required steps:")
    for number, step in enumerate(task.required_steps, 1):
        print(f"   {number}. {step}")
    print(f"\nevaluation rubric:\n{task.evaluation}")

    files = task.inputs()
    print(f"{len(files)} staged inputs; deliverable goes to {task.output_path!r}")
    for f in files[:4]:
        print(f"   {f['path']:<58} {f.get('size')} bytes")

    head = task.fetch_input(files[0]["path"])[:200].decode("utf-8", "replace")
    print(f"\nfirst input begins: {head!r}")

    task.submit_text(task.output_path or "output/agent_output.json", '{"stub": true}')
    g = task.grade(keep_alive=True)
    print(f"\nscore {g.pass_rate}  overall_success {g.overall_success}")
    print("per-verifier results:")
    for v in g.per_verifier:
        print(v["name"], v.get("score"), v["passed"])
        for check in (v.get("details") or {}).get("checks", []):
            print("   -", check["name"], check.get("score"), check["passed"])

## 6. Persistent memory

`memory_key` gives a Codex run a persistent `CODEX_HOME` on the service, so rollouts accumulate
across runs under a key you choose. `consolidate_memory(key)` then asks Codex to distil those
rollouts into its memory file — the pair is a working memory-based self-evolving method.

```python
acp_codex_agent(task, api_key=..., memory_key="my-run-42")     # accumulate
client.consolidate_memory("my-run-42", label="after-stage-1")  # distil
```

The natural place for this is Level 2's `adapt(stage, tasks)`: run the stage's training tasks
with a `memory_key`, consolidate at the end of the stage, and the evaluation that follows sees
the distilled memory. `memory_main_only` restricts reading to the main memory file;
`root_only` restricts consolidation to the root home.

Memory is **server-side state keyed by a string you pick**, so choose keys that will not collide
with your own other runs, and treat a key as the unit you reset when you want a clean experiment.

## 7. Long runs

Agent runs can outlast an HTTP connection, so the SDK does not rely on one. `run_agent` and
`run_codex` are submitted with `background=true` and polled to completion — the request that
started the run can drop without killing it, and a proxy's idle timeout cannot turn a working
run into a spurious `5xx`.

You get this by default; there is nothing to enable. What is worth knowing:

- **`timeout_s`** is the real budget, enforced server-side. For Codex it is shared across all
  `max_episodes`, not per episode.
- **Sessions are reaped when idle**, but never while a background run is in flight.
- **A job's result is polled once.** `status` moves `pending → running → done | error`; on error
  you get the `{status_code, detail}` the synchronous call would have raised.
- **`run_benchmark` is a client-side loop**, so it is *not* backgrounded as a whole. A full-scope
  sweep runs for hours in your process — run it somewhere it will not be interrupted, and
  remember `on_error="score_zero"` keeps a long sweep alive through individual failures.

## 8. Observability

| | |
|---|---|
| `client.health()` | liveness, active sessions, TTL, ALE docker + task-set status |
| `client.usage(top=20)` | server-side usage attributed to your key |
| `GET /dashboard` | the same, rendered — open it in a browser |
| client logging | every call and its latency |

The service records usage centrally, so `client.usage()` is the authority on what you spent
against it — distinct from your OpenAI spend, which only your provider sees.

In [ ]:
import logging

h = client.health()
print("health:", {k: h.get(k) for k in ("ok", "active_sessions", "ttl_sec")})
print("ale:   ", (h.get("ale_tasks") or {}).get("n_runnable"), "runnable tasks")

u = client.usage(top=5)
print("\nusage keys:", list(u)[:8])

# Turn on per-call logging when you want to see what the SDK is actually doing.
logging.basicConfig(level=logging.INFO)
logging.getLogger("simple_agentic_evals").setLevel(logging.DEBUG)
client.benchmarks()
logging.getLogger("simple_agentic_evals").setLevel(logging.WARNING)   # back to quiet

## 9. Raw HTTP — no SDK

The SDK is a convenience over a plain REST API. Everything is reachable with `curl`, which is
what you want from another language, or when debugging what the SDK sent.

Auth is `Authorization: Bearer <EVAL_SERVICE_API_KEY>` on every `/v1/*` route, health included.

```bash
BASE=https://…; KEY=…
curl -s -H "Authorization: Bearer $KEY" $BASE/v1/health
curl -s -H "Authorization: Bearer $KEY" "$BASE/v1/tasks?dataset=evovling_tools&benchmark=eog&version=1&split=test&domain=hr"
curl -s -X POST -H "Authorization: Bearer $KEY" -H 'Content-Type: application/json' \
     -d '{"dataset":"evovling_tools","benchmark":"eog","version":1,"split":"test","domain":"hr","task_id":"…"}' \
     $BASE/v1/sessions
curl -s -X POST -H "Authorization: Bearer $KEY" "$BASE/v1/sessions/$SID/grade?keep_alive=true"
```

The MCP endpoint is a standard streamable-HTTP MCP server at
`/v1/sessions/{id}/mcp/{server_name}` — point any MCP client at it, as Level 1 §2 shows.

In [ ]:
import urllib.request

req = urllib.request.Request(
    f"{SERVICE_URL}/v1/health",
    headers={"Authorization": f"Bearer {os.environ['EVAL_SERVICE_API_KEY']}",
             "ngrok-skip-browser-warning": "true"})
print(json.load(urllib.request.urlopen(req, timeout=30)).get("ok"))

# Every endpoint, straight from the running service's OpenAPI schema.
spec = json.load(urllib.request.urlopen(urllib.request.Request(
    f"{SERVICE_URL}/openapi.json", headers={"ngrok-skip-browser-warning": "true"})))
for path, ops in sorted(spec["paths"].items()):
    print(f"  {'/'.join(m.upper() for m in ops):<12} {path}")

## 10. API reference

### `EvalClient`

| | |
|---|---|
| `EvalClient(base_url=None, api_key=None)` | reads `$EVAL_SERVICE_URL` / `$EVAL_SERVICE_API_KEY`; the wheel knows its own service |
| `.health()` | liveness + capability detail |
| `.usage(top=20)` | your usage against this service |
| `.benchmarks()` | the catalog: datasets → benchmarks → domains → versions with `n_train`/`n_test` |
| `.task_ids(dataset, benchmark, version, split, domain, limit, offset)` | ids only |
| `.tasks(...)` → `Iterator[Task]` | lazy handles; add `resource_mode=` |
| `.task(dataset, benchmark, version, task_id, split, domain, resource_mode)` | one handle |
| `.resources(dataset, benchmark, version, task_id, split, domain, mode, include_content)` | the evolving pool, no session needed |
| `.consolidate_memory(memory_key, label, model, api_key, root_only)` | distil a memory home |

### `Task`

| lifecycle | |
|---|---|
| `with task:` / `.start()` / `.close()` | provision and tear down |
| `.grade(keep_alive=False)` → `GradeResult` | run the verifiers |
| `.evaluate(agent, keep_alive, fetch_artifacts, **kw)` → `EvalReport` | provided harness + grade |

| attributes | |
|---|---|
| `.task_id` `.benchmark` `.dataset` `.action_type` | identity |
| `.system_prompt` `.user_prompt` | populated by `start()`, not before |
| `.oracle_tools` `.resources` | the gold set; the attached evolving pool |
| `.session_id` | live session, or `None` |

| EOG | |
|---|---|
| `.mcp_servers` → `[McpServer]` | `.name` `.url` `.headers` `.transport` `.path` |
| `.mcp_url(server)` | absolute MCP endpoint — hand to any MCP client |
| `.mcp_session(server, timeout)` → `MCPSession` | `.list_tools()` `.call_tool(name, args)` `.close()` |

| ALE | |
|---|---|
| `.inputs()` `.fetch_input(p)` `.fetch_inputs_to(dir)` | staged inputs |
| `.output_path` | the expected deliverable |
| `.submit(files)` `.submit_text(p, s)` `.submit_dir(dir, max_bytes=32MB)` | the artifact |
| `.fetch_run_artifacts()` | tar.gz of the `ale_run` tree — call **before** grading |

### Results

| `GradeResult` | `pass_rate` `overall_success` `n_passed` `n_total` `per_verifier` |
|---|---|
| **`EvalReport`** | `accuracy` `overall_success` `latency_s` `total_tokens` `input_tokens` `output_tokens` `cache_read_tokens` `cost_usd` `n_steps` `n_subagent_spawns` `grade` `run` `artifacts_tgz` |
| **`CodexRun`** | `episodes` `stopped` `completed` `exit_code` `thread_id` `n_calls` `n_exec` `final_message` `latency_s` `total_tokens` `n_subagent_spawns`, plus ALE-only `cache_read_tokens` `cost_usd` `n_steps` |
| **`AgentRun`** | `n_calls` `n_exec` `steps` `stopped` `completed` `final_message` `latency_s` `total_tokens` `input_tokens` `output_tokens` `tools_used` `messages`, and `trace` — one dict per step of `{thought, tool_calls, tool_results, usage}` |

To inspect every task × verifier row after `run_benchmark(...)`:

```python
for result in report.verifier_results:
    print(result["task_id"], result["name"], result["score"], result["passed"])
```

### `run_benchmark`

| | |
|---|---|
| `agent` | `f(task)`, or `"react"` / `"codex"` |
| `mode` | `deployment_eval` \| `self_evolving_adapt_eval` \| `task_specific` |
| `dataset` `benchmark` `domain` `split` | scope |
| `adapt=fn(stage, tasks)` | per-stage learning; or an `.adapt` method |
| `matrix=True` | fill the triangle → BWT / FWT |
| `limit` `agent_kwargs` `on_error` `client` `progress` | the rest |

| `BenchmarkReport` | |
|---|---|
| `accuracy` `success_rate` | **Score** and **Pass %** |
| `agent_s` `n_with_latency` | agent time (the comparable one); `latency_s` is wall clock |
| `total_tokens` `n_with_tokens` `tokens_per_task` | `None` when unmeasured, never a false 0 |
| `n_tasks` `n_success` `n_errors` `complete` `paper_n_tasks` | scope and honesty |
| `last_row` `matrix` `before_adapt` | `CohortResult` lists |
| `bwt` `fwt` `per_stage_acc` | transfer metrics |
| `summary()` / `print(report)` | the paper-style block |

| `CohortResult` | `domain` `stage` `at_stage` `n_tasks` `accuracy` `success_rate` `n_errors` `agent_s` `total_tokens` `per_task` |
|---|---|

### Harnesses, metrics, errors

| | |
|---|---|
| `react_agent(task, model, api_key, max_steps, restrict_to_selected_tools, timeout_s, verbose, include_trace)` | EOG only |
| `acp_codex_agent(task, model, api_key, transport, allowed_tools, mcp_only, max_episodes, timeout_s, require_completion, completion_sentinel, verbose, include_trace, prompt_suffix, sandbox_env, memory_key, memory_main_only)` | EOG + ALE |
| `to_openai_tools(mcp_tools)` / `sanitize_tool_schema(schema)` | MCP → OpenAI |
| `ContinualMetrics(num_stages)` / `StageResult(...)` | build a matrix yourself (0-based; `adapt_stage=-1` is the FWT baseline) |
| `ServiceError` | `.status_code` `.detail` |
| `MissingAPIKey` | raised client-side before any request; `run_benchmark` re-raises it rather than scoring zero |
| `PAPER_SCOPE` / `NON_PAPER_EOG_DOMAINS` | published denominators; the domain excluded from them |